### Classification benchmark template (have data optimization/processing in a separate file)

**See the "modules_to_install.txt" for needed pip installations.**

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import tensorflow as tf
import keras
from keras import layers

from sklearn.preprocessing import label_binarize


### Phase 1: Load the data

**To keep this benchmarking notebook more simple, have data processing/optimization etc. in a separate notebook, and save the final optimized version into a new csv-file which you will use here.**

In [2]:
# load the credit group data, replace this with your own .csv -data
# this version of data should be READY TO GO to training
# which means => only numeric values in the dataset
df = pd.read_csv("processed_classification_credit_group.csv")
df.head()

,Delay_from_due_date,Num_of_Delayed_Payment,Num_Credit_Inquiries,Credit_Utilization_Ratio,Credit_History_Age,Amount_invested_monthly,Monthly_Balance,Credit_Mix,Age,Annual_Income,...,Outstanding_Debt,Total_EMI_per_month,Payment_of_Min_Amount_No,Payment_of_Min_Amount_Yes,Payment_Behaviour_High_spent_Large_value_payments,Payment_Behaviour_High_spent_Medium_value_payments,Payment_Behaviour_High_spent_Small_value_payments,Payment_Behaviour_Low_spent_Large_value_payments,Payment_Behaviour_Low_spent_Medium_value_payments,Credit_Score
0,3.0,7.0,4.0,26.822620,265.0,80.415295,312.494089,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2
1,3.0,7.0,4.0,31.944960,265.0,118.280222,284.629162,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2
2,3.0,7.0,4.0,28.609352,267.0,81.699521,331.209863,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2
3,5.0,4.0,4.0,31.377862,268.0,199.458074,223.451310,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2
4,6.0,4.0,4.0,24.797347,269.0,41.420153,341.489231,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2


### X/y + train/test -splits + other data setups

**Fix the categoricals, continuous and target variable according to your own dataset.**

In [3]:
target = "Credit_Score"

# CatBoost requires we save our continuous and categorical variables separately into lists
categorical_variables = ["Credit_Mix", 'Payment_of_Min_Amount_No', 'Payment_of_Min_Amount_Yes',
                        'Payment_Behaviour_High_spent_Large_value_payments',
                        'Payment_Behaviour_High_spent_Medium_value_payments',
                        'Payment_Behaviour_High_spent_Small_value_payments',
                        'Payment_Behaviour_Low_spent_Large_value_payments',
                        'Payment_Behaviour_Low_spent_Medium_value_payments']

# the usual X/y -split
X = df.drop(target, axis=1)
y = df[target]

# usual train/test -split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# NOTE! SCALING => some of the algorithms require this
scaler = StandardScaler()

# create separate versions for the scaled data
# because we need both unscaled and scaled versions later
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### List all our models that we want to benchmark

In [4]:
# define our model test dictionary

# we have quite a lot of data ~50000 rows
# keep in mind => this amount of data might be too heavy for KNN

# for the hyperparameter optimization results, see
# "classification_hyperparameter_optimization.ipynb"

# RandomizedSearchCV results CatBoost:
# Best Parameters: {'learning_rate': np.float64(0.14), 'iterations': np.float64(600.0), 'depth': 7}

# we disabled SVM and logistic regression, because they are not very suitable
# for a large multiclass classification dataset like this
# SVM especially is super slow here.

models = {
    # "Logistic Regression": LogisticRegression(solver="saga"),
    # "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "XGBoost": xgb.XGBClassifier(enable_categorical=True, objective="multi:softprob", num_class=len(np.unique(y_train))),
    "CatBoost-default": cb.CatBoostClassifier(iterations=500, verbose=0),
    "CatBoost-opt": cb.CatBoostClassifier(learning_rate=np.float64(0.14), iterations=np.float64(600.0), depth=7, verbose=0),
    'LightGBM': lgb.LGBMClassifier(objective='multiclass')

}

### We need a for-loop to train all the models in one go + gather metrics

**This part is a bit tricky with classification due to how probabilities are handled**

In [5]:
# our benchmarking code!

# this will contain our results later after the benchmarks
results = []

# loop through the models
for name, model in models.items():
    # we have to react to certain algorithms
    # because they have special requirements for fit()

    print("Starting..." + name)

    # SVM / KNN require scaled data
    if name in ["SVM", "KNN"]:
        model.fit(X_train_scaled, y_train)
        predictions = model.predict(X_test_scaled)

        # if there is a predict_proba -method available
        # in this algorithm => use it, otherwise return None
        probability = model.predict_proba(X_test_scaled) if hasattr(model, "predict_proba") else None
    elif name == "CatBoost":
        # CatBoost requires we specify which columns are categories
        # this includes both ordinal and nominal categories
        model.fit(X_train, y_train, cat_features=categorical_variables)
        predictions = model.predict(X_test)
        probability = model.predict_proba(X_test)
    else:
        # all the other algorithms
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        # use predict_proba if available, otherwise return None
        probability = model.predict_proba(X_test) if hasattr(model, "predict_proba") else None

    # after training this particular algorithm, gather the metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, average="macro")
    recall = recall_score(y_test, predictions, average="macro")
    f1 = f1_score(y_test, predictions, average="macro")

    # ROC AUC, we need the probability here
    if probability is not None:
        # the classes list [0, 1, 2] => this should be replace
        # with some more dynamic code from data itself
        y_test_bin = label_binarize(y_test, classes=[0, 1 ,2])
        roc_auc = roc_auc_score(y_test, probability, multi_class="ovr")
    else:
        roc_auc = np.nan

    # save the metrics for this model into the results
    results.append([name, accuracy, precision, recall, f1, roc_auc])

Starting...KNN
Starting...Random Forest
Starting...XGBoost
Starting...CatBoost-default
Starting...CatBoost-opt
Starting...LightGBM
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000695 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2537
[LightGBM] [Info] Number of data points in the train set: 65869, number of used features: 25
[LightGBM] [Info] Start training from score -1.285313
[LightGBM] [Info] Start training from score -0.632891
[LightGBM] [Info] Start training from score -1.648273


### Let's visualize the results

**This code was originally created with the help of ChatGPT**

In [6]:
metrics_df = pd.DataFrame(results, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC'])

# change these highlight colors as you wish
# default style is dart theme / mode, modify
# variable default_highlight_style if you wish 
# to use light theme instead
light_theme = 'background: yellow'
dark_theme = 'background: #8C670A'

# you can alternate the highlight styling based on your theme
default_highlight_style = dark_theme

# helper function that highlights the best model of each metric
def highlight_best_metrics(row):
    # default styles for everything is empty in the beginning
    styles = ['' for _ in row]

    # index 1 => Accuracy (index 0 => model name)
    if row['Accuracy'] == metrics_df['Accuracy'].max():
        styles[1] = default_highlight_style

    # index 2 = > Precision
    if row['Precision'] == metrics_df['Precision'].max():
        styles[2] = default_highlight_style

    # index 3 = > Recall
    if row['Recall'] == metrics_df['Recall'].max():
        styles[3] = default_highlight_style

    # index 4 = > F1-score
    if row['F1-score'] == metrics_df['F1-score'].max():
        styles[4] = default_highlight_style

    # index 5 = > ROC-AUC
    if row['ROC-AUC'] == metrics_df['ROC-AUC'].max():
        styles[5] = default_highlight_style

    return styles


# apply the custom styles based on the min/max metrics
highlight_df = metrics_df.style.apply(highlight_best_metrics, axis=1)


highlight_df

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,KNN,0.801069,0.787978,0.779783,0.783692,0.912337
1,Random Forest,0.876123,0.870939,0.869867,0.870319,0.958165
2,XGBoost,0.830641,0.818861,0.819975,0.819256,0.930891
3,CatBoost-default,0.819650,0.804062,0.808630,0.805823,0.923364
4,CatBoost-opt,0.837442,0.824706,0.827143,0.825797,0.932110
5,LightGBM,0.799733,0.782529,0.789911,0.784565,0.915237


In [7]:
# Random Forest seems to work well here!